# MergeSort and Merge (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/MergeSort_cpp.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `MergeSort` and `Merge`: the algorithms in C++, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/MergeSort_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/MergeSort_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `RecursiveSum` (PascalCase).

## Merge Sort and Merge

- **MergeSort**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the sub-array $a[l..r]$ sorted in non-decreasing order. It is called `MergeSort(a, 1, n)` in the pseudo-code, `merge_sort(a, 0, n-1)` in the code.
- **Merge**$(a,l,m,r)$ — *Input*: an array and indices $l\le m<r$ such that $a[l..m]$ and $a[m+1..r]$ are sorted. *Output*: $a[l..r]$ sorted. It copies the two halves into $L$ and $R$ and moves back into $a$, step by step, the smaller of the first two elements not yet copied.

The counter `comparisons` counts the comparisons between elements executed by `Merge`. On the array of the example $a=[9,12,3,16,19,11,12]$ we print the calls of `Merge` in execution order (with the array after each of them, table of the chapter; indices are from 0: adding 1 gives those of the chapter), then the trace of `Merge(a, 0, 1, 3)` on $a=[5,8,2,7]$ (example of the C++ implementation).

In [1]:
%%writefile merge_sort.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long comparisons = 0;   // comparisons between elements executed by Merge
int msCalls = 0;             // calls of MergeSort

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Merge(a, l, m, r): merges the sorted a[l..m] and a[m+1..r]
void Merge(int *a, int l, int m, int r)
{
    int n1 = m - l + 1;             // size of the left part
    int n2 = r - m;                 // size of the right part
    vector<int> L(n1), R(n2);       // auxiliary arrays
    for (int i = 0; i < n1; i++)
    {
        L[i] = a[l + i];
    }
    for (int j = 0; j < n2; j++)
    {
        R[j] = a[m + 1 + j];
    }
    int i = 0, j = 0, k = l;
    while (i < n1 && j < n2)        // merge L and R into a[l..r]
    {
        comparisons++;
        if (L[i] <= R[j])
        {
            a[k] = L[i];
            i++;
        }
        else
        {
            a[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1)                  // copy the rest of L
    {
        a[k] = L[i];
        i++;
        k++;
    }
    while (j < n2)                  // copy the rest of R
    {
        a[k] = R[j];
        j++;
        k++;
    }
}

// MergeSort(a, l, r): sorts a[l..r]
void MergeSort(int *a, int l, int r)
{
    msCalls++;
    if (l < r)
    {
        int m = l + (r - l) / 2;    // divide
        MergeSort(a, l, m);         // conquer (left half)
        MergeSort(a, m + 1, r);     // conquer (right half)
        Merge(a, l, m, r);          // combine
    }
}

// ---- variants that print the trace ----
void MergeSortLog(int *a, int l, int r)
{
    if (l < r)
    {
        int m = l + (r - l) / 2;
        MergeSortLog(a, l, m);
        MergeSortLog(a, m + 1, r);
        string left = Fmt(a + l, m - l + 1), right = Fmt(a + m + 1, r - m);
        Merge(a, l, m, r);
        printf("Merge(a,%d,%d,%d)  %s and %s  ->  %s\n", l, m, r, left.c_str(), right.c_str(), Fmt(a, 7).c_str());
    }
}

void MergeTrace(int *a, int n, int l, int m, int r)
{
    int n1 = m - l + 1, n2 = r - m;
    vector<int> L(a + l, a + m + 1), R(a + m + 1, a + r + 1);
    printf("Merge(a, %d, %d, %d) on a = %s\n", l, m, r, Fmt(a, n).c_str());
    printf("n1=%d, n2=%d, L=%s, R=%s\n", n1, n2, Fmt(L.data(), n1).c_str(), Fmt(R.data(), n2).c_str());
    int i = 0, j = 0, k = l, it = 0;
    while (i < n1 && j < n2)
    {
        it++;
        char head[200];
        if (L[i] <= R[j])
        {
            snprintf(head, sizeof head, "%2d. loop 1: L[%d]=%d <= R[%d]=%d? Yes => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
            a[k] = L[i];
            i++;
        }
        else
        {
            snprintf(head, sizeof head, "%2d. loop 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
            a[k] = R[j];
            j++;
        }
        k++;
        printf("%s, i=%d, j=%d, k=%d; a=%s\n", head, i, j, k, Fmt(a, n).c_str());
    }
    while (i < n1)
    {
        a[k] = L[i];
        printf("    loop 2 (rest of L): a[%d] = L[%d] = %d; a=%s\n", k, i, L[i], Fmt(a, n).c_str());
        i++;
        k++;
    }
    while (j < n2)
    {
        a[k] = R[j];
        printf("    loop 3 (rest of R): a[%d] = R[%d] = %d; a=%s\n", k, j, R[j], Fmt(a, n).c_str());
        j++;
        k++;
    }
    printf("result a = %s, comparisons = %d\n", Fmt(a, n).c_str(), it);
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 12, 3, 16, 19, 11, 12};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    printf("Calls of Merge in execution order:\n");
    int copia[] = {9, 12, 3, 16, 19, 11, 12};
    MergeSortLog(copia, 0, n - 1);
    comparisons = 0;
    msCalls = 0;
    MergeSort(a, 0, n - 1);
    printf("sorted = %s   calls of MergeSort=%d (2n-1=%d)   comparisons=%lld\n", Fmt(a, n).c_str(), msCalls, 2 * n - 1, comparisons);
    printf("\n");
    int b[] = {5, 8, 2, 7};
    MergeTrace(b, 4, 0, 1, 3);
    return 0;
}
#endif

Writing merge_sort.cpp


In [2]:
!g++ -std=c++17 -O2 -o merge_sort merge_sort.cpp && ./merge_sort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Calls of Merge in execution order:
Merge(a,0,0,1)  [9] and [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] and [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] and [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] and [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] and [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] and [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
sorted = [3, 9, 11, 12, 12, 16, 19]   calls of MergeSort=13 (2n-1=13)   comparisons=14

Merge(a, 0, 1, 3) on a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. loop 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. loop 1: L[0]=5 <= R[1]=7? Yes => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. loop 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    loop 2 (rest of L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
result a = [2, 5, 7, 8], comparisons = 3


**Try it yourself.** Sort another array: the number of comparisons of `Merge` is at most $n\lceil\log_2 n\rceil$ and the calls of `MergeSort` are always $2n-1$.

In [3]:
%%writefile prova_merge_sort.cpp
#define NOMAIN
#include "merge_sort.cpp"

int main()
{
    // Sort another array and count the comparisons (at most n*ceil(log2 n)).
    int b[] = {38, 27, 43, 3, 9, 82, 10, 5};
    int n = 8;
    comparisons = 0;
    msCalls = 0;
    MergeSort(b, 0, n - 1);
    printf("%s %lld %d\n", Fmt(b, n).c_str(), comparisons, msCalls);
    return 0;
}

Writing prova_merge_sort.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_merge_sort prova_merge_sort.cpp && ./prova_merge_sort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).